In [1]:
1

1

In [2]:
import json
import shutil
from pathlib import Path

import anndata as ad
import numpy as np
import os
import pandas as pd

In [3]:
from tqdm import tqdm

In [4]:
import pandas as pd

In [5]:
PLIBDATA_ROOT = '../.plib_cache/raw_datasets'

In [6]:
cols = ['dataset', 'context', 'perturbation', 'log_dose', 'time']

In [7]:
SHARDSIZE = 200_000
VALDATA_PORTION = 0.15
TESTDATA_PORTION = 0.15
SPLIT_SEED = 13

In [8]:
df_annot_all = []
for i, folder in tqdm(enumerate(os.listdir(PLIBDATA_ROOT))):
    for j, file in enumerate(os.listdir(f'{PLIBDATA_ROOT}/{folder}')):
        df_annot = pd.read_parquet(f'{PLIBDATA_ROOT}/{folder}/{file}')[cols].drop_duplicates()
        df_annot_all.append(df_annot)
        
pd.concat(df_annot_all).to_parquet('../.plib_cache/annotations/df_annot_all.parquet', index=False)

9it [05:34, 37.21s/it]


In [9]:
pd.concat(df_annot_all)

,dataset,context,perturbation,log_dose,time
0,CIGS MCE,CVCL_0063,5281081,1.000000,24.0
2881,CIGS MCE,CVCL_0063,11957668,1.000000,24.0
5762,CIGS MCE,CVCL_0063,127264445,1.000000,24.0
8643,CIGS MCE,CVCL_0063,60651,1.000000,24.0
11524,CIGS MCE,CVCL_0063,6034,1.000000,24.0
...,...,...,...,...,...
1736466,Ginkgo GDPx2,CL_0002551,5702062,-0.522879,24.0
1746864,Ginkgo GDPx2,CL_0002551,49867937,-0.522879,24.0
1757262,Ginkgo GDPx2,CL_0002551,5702062,-2.022276,24.0
1767660,Ginkgo GDPx2,CL_0002551,16088020,-2.022276,24.0


In [10]:
df_annot_all_ = pd.read_parquet('../.plib_cache/annotations/df_annot_all.parquet')

In [11]:
df_annot_all_['context_perturbation'] = df_annot_all_['context'].astype(str) + '_' + df_annot_all_['perturbation'].astype(str)

In [12]:
df_annot_all_ = df_annot_all_.drop_duplicates(['dataset','context_perturbation'])

In [13]:
df_annot_all_

,dataset,context,perturbation,log_dose,time,context_perturbation
0,CIGS MCE,CVCL_0063,5281081,1.000000,24.0,CVCL_0063_5281081
1,CIGS MCE,CVCL_0063,11957668,1.000000,24.0,CVCL_0063_11957668
2,CIGS MCE,CVCL_0063,127264445,1.000000,24.0,CVCL_0063_127264445
3,CIGS MCE,CVCL_0063,60651,1.000000,24.0,CVCL_0063_60651
4,CIGS MCE,CVCL_0063,6034,1.000000,24.0,CVCL_0063_6034
...,...,...,...,...,...,...
340593,Ginkgo GDPx2,CL_0002551,39484,0.477121,24.0,CL_0002551_39484
340594,Ginkgo GDPx2,CL_0002551,201899,-1.022276,24.0,CL_0002551_201899
340596,Ginkgo GDPx2,CL_0002551,16667669,0.477121,24.0,CL_0002551_16667669
340597,Ginkgo GDPx2,CL_0002551,49867937,0.477121,24.0,CL_0002551_49867937


In [14]:
# step A: build pair-level table. the split unit is the (context, perturbation) PAIR, not
# the (dataset, context, perturbation) ROW. this guarantees that every row sharing a pair
# -- across all datasets -- lands in the same split, eliminating pair-level train/val/test
# leaks. dataset multiplicity is tracked per pair so per-dataset holdout budgets can be
# enforced later in step B.
HOLDOUT_PORTION = VALDATA_PORTION + TESTDATA_PORTION

pair_df = (
    df_annot_all_
    .drop_duplicates('context_perturbation')
    [['context_perturbation', 'context', 'perturbation']]
    .reset_index(drop=True)
)
pair_datasets = (
    df_annot_all_
    .groupby('context_perturbation')['dataset']
    .agg(lambda s: tuple(sorted(set(s))))
    .to_dict()
)
pair_df['datasets']   = pair_df['context_perturbation'].map(pair_datasets)
pair_df['n_datasets'] = pair_df['datasets'].map(len)
pair_df['split']      = 'train'

print(f'total pairs: {len(pair_df)}')
print('pairs by dataset multiplicity:')
print(pair_df['n_datasets'].value_counts().sort_index())

total pairs: 154128
pairs by dataset multiplicity:
n_datasets
1    146963
2      7014
3       151
Name: count, dtype: int64


In [15]:
# step B: per-dataset budgeted greedy holdout selection. multi-dataset pairs are placed
# FIRST (hardest to fit; they consume budget in every dataset they touch); single-dataset
# pairs fill the remainder. tie-break is random and seeded.
#
# a pair is accepted into holdout only if all three constraints hold:
#   1. its per-dataset cap is not exceeded in ANY of its datasets,
#   2. removing it from train still leaves at least one train pair with its context,
#   3. removing it from train still leaves at least one train pair with its perturbation.
# the context- and perturbation-level learnability checks are dynamic: counters are updated
# as pairs are accepted, so each decision reflects the current state of the train pool.
rng = np.random.RandomState(SPLIT_SEED)

ctx_train_count  = pair_df['context'].value_counts().to_dict()
pert_train_count = pair_df['perturbation'].value_counts().to_dict()

dataset_total        = df_annot_all_.groupby('dataset').size().to_dict()
dataset_holdout_cap  = {d: int(n * HOLDOUT_PORTION) for d, n in dataset_total.items()}
dataset_holdout_used = {d: 0 for d in dataset_total}

pair_df['rand'] = rng.random(len(pair_df))
order = pair_df.sort_values(
    ['n_datasets', 'rand'],
    ascending=[False, True],
).index.to_numpy()

pair_split = {p: 'train' for p in pair_df['context_perturbation']}
n_accept = n_rej_budget = n_rej_ctx = n_rej_pert = 0

for i in order:
    c   = pair_df.at[i, 'context']
    p   = pair_df.at[i, 'perturbation']
    ds  = pair_df.at[i, 'datasets']
    pid = pair_df.at[i, 'context_perturbation']

    if ctx_train_count[c] <= 1:
        n_rej_ctx += 1
        continue
    if pert_train_count[p] <= 1:
        n_rej_pert += 1
        continue
    if any(dataset_holdout_used[d] + 1 > dataset_holdout_cap[d] for d in ds):
        n_rej_budget += 1
        continue

    pair_split[pid] = 'holdout'
    ctx_train_count[c]  -= 1
    pert_train_count[p] -= 1
    for d in ds:
        dataset_holdout_used[d] += 1
    n_accept += 1

pair_df['split'] = pair_df['context_perturbation'].map(pair_split)
pair_df.drop(columns=['rand'], inplace=True)

print(f'pairs accepted to holdout: {n_accept}')
print(f'  rejected - dataset budget exhausted: {n_rej_budget}')
print(f'  rejected - last train pair for ctx:  {n_rej_ctx}')
print(f'  rejected - last train pair for pert: {n_rej_pert}')
print(pair_df['split'].value_counts())

pairs accepted to holdout: 41283
  rejected - dataset budget exhausted: 97200
  rejected - last train pair for ctx:  0
  rejected - last train pair for pert: 15645
split
train      112845
holdout     41283
Name: count, dtype: int64


In [16]:
# step C: partition holdout into val/test by PERTURBATION (val perts disjoint from test
# perts across all datasets), then broadcast the pair-level split to every (dataset, pair)
# row. because the assignment is keyed on the pair, all dataset-rows of a given pair share
# the same split label -- which is what eliminates the pair-level leak.
rng_vt = np.random.RandomState(SPLIT_SEED + 5000)

holdout_perts = pair_df.loc[pair_df['split'] == 'holdout', 'perturbation'].unique()
rng_vt.shuffle(holdout_perts)
n_val_perts = int(round(len(holdout_perts) * VALDATA_PORTION / HOLDOUT_PORTION))
val_perts  = set(holdout_perts[:n_val_perts])
test_perts = set(holdout_perts[n_val_perts:])

mask_val  = (pair_df['split'] == 'holdout') & pair_df['perturbation'].isin(val_perts)
mask_test = (pair_df['split'] == 'holdout') & pair_df['perturbation'].isin(test_perts)
pair_df.loc[mask_val,  'split'] = 'val'
pair_df.loc[mask_test, 'split'] = 'test'

assert (pair_df['split'] != 'holdout').all(), 'holdout label still present'
assert val_perts.isdisjoint(test_perts), 'val and test perturbations overlap'

pair_to_split = dict(zip(pair_df['context_perturbation'], pair_df['split']))
df_annot_split = df_annot_all_.copy()
df_annot_split['split'] = df_annot_split['context_perturbation'].map(pair_to_split)

print(f'holdout perturbations split: {len(val_perts)} -> val, {len(test_perts)} -> test')
print('pair-level split:')
print(pair_df['split'].value_counts())
print('row-level split:')
print(df_annot_split['split'].value_counts())

holdout perturbations split: 8828 -> val, 8829 -> test
pair-level split:
split
train    112845
val       20709
test      20574
Name: count, dtype: int64
row-level split:
split
train    114137
val       23661
test      23646
Name: count, dtype: int64


In [17]:
# sanity checks on the final split:
#   1) learnability: every perturbation and context in val/test should also appear in train
#      somewhere (so its embedding receives gradient during training).
#   2) val/test perturbation disjointness: a perturbation must not appear in BOTH val and
#      test. (sharing with train is allowed and necessary for learnability; sharing across
#      val/test would mean the model has seen the pert in val by the time we evaluate on
#      test.)
#   3) PAIR-level disjointness: no (context, perturbation) pair appears in more than one
#      split. this is the leak the new pair-level strategy is designed to prevent and is
#      the key check this strategy adds over the previous implementation.
pert_train = set(df_annot_split.loc[df_annot_split['split'] == 'train', 'perturbation'])
ctx_train  = set(df_annot_split.loc[df_annot_split['split'] == 'train', 'context'])
pert_val   = set(df_annot_split.loc[df_annot_split['split'] == 'val',   'perturbation'])
pert_test  = set(df_annot_split.loc[df_annot_split['split'] == 'test',  'perturbation'])
ctx_holdout = set(df_annot_split.loc[df_annot_split['split'].isin(['val', 'test']), 'context'])

pairs_train = set(df_annot_split.loc[df_annot_split['split'] == 'train', 'context_perturbation'])
pairs_val   = set(df_annot_split.loc[df_annot_split['split'] == 'val',   'context_perturbation'])
pairs_test  = set(df_annot_split.loc[df_annot_split['split'] == 'test',  'context_perturbation'])

unseen_pert = (pert_val | pert_test) - pert_train
unseen_ctx  = ctx_holdout - ctx_train

print(f'perturbations only in val/test (no train pair anywhere): {len(unseen_pert)}')
print(f'contexts      only in val/test (no train pair anywhere): {len(unseen_ctx)}')
print(f'perturbations shared between val and test:               {len(pert_val & pert_test)}')
print(f'pairs in BOTH train and val :                            {len(pairs_train & pairs_val)}')
print(f'pairs in BOTH train and test:                            {len(pairs_train & pairs_test)}')
print(f'pairs in BOTH val   and test:                            {len(pairs_val   & pairs_test)}')

perturbations only in val/test (no train pair anywhere): 0
contexts      only in val/test (no train pair anywhere): 0
perturbations shared between val and test:               0
pairs in BOTH train and val :                            0
pairs in BOTH train and test:                            0
pairs in BOTH val   and test:                            0


In [18]:
report = (
    df_annot_split.groupby(['dataset', 'split']).size()
    .unstack(fill_value=0)
    .reindex(columns=['train', 'val', 'test'], fill_value=0)
)
report['total'] = report.sum(axis=1)
for c in ['train', 'val', 'test']:
    report[f'{c}_frac'] = (report[c] / report['total']).round(3)
report

split,train,val,test,total,train_frac,val_frac,test_frac
dataset,,,,,,,
CIGS MCE,15274,3264,3281,21819,0.700,0.150,0.150
CIGS TCM,2527,523,559,3609,0.700,0.145,0.155
Ginkgo GDPx2,241,49,54,344,0.701,0.142,0.157
Ginkgo VCPI vcpi-0001 (tvc-bhr-009),2269,3,0,2272,0.999,0.001,0.000
Ginkgo VCPI vcpi-0002 (tvc-kdl-010),1487,0,1,1488,0.999,0.000,0.001
LINCS_phase1_level3_epsilon,80858,17410,17243,115511,0.700,0.151,0.149
LINCS_phase2_level3,10914,2294,2383,15591,0.700,0.147,0.153
dilimap_train,175,40,35,250,0.700,0.160,0.140
srivatsan20_sciplex3,392,78,90,560,0.700,0.139,0.161


In [19]:
df_annot_split.columns

Index(['dataset', 'context', 'perturbation', 'log_dose', 'time',
       'context_perturbation', 'split'],
      dtype='object')

In [20]:
df_annot_split[['dataset', 'context', 'perturbation', 'log_dose', 'time', 'split']].to_parquet(
        '../.plib_cache/annotations/df_annot_split.parquet',
        index=False,
        )